# Interactive Detection Gap Explorer

## Part 5 of the GSMNP SLF Detection Gap Project

The three-panel static map from notebook 04 shows the habitat suitability, observer effort, and detection gap pattern clearly — but it can't be zoomed, queried by location, or used to plan a field route. This notebook builds an interactive version using Folium, layered on a web basemap.

Two additions appear here that the static map doesn't include: streams and water bodies from the National Hydrography Dataset (NHD), which give geographic grounding for where high-gap areas sit relative to natural corridors.

Click anywhere on the map to see the latitude and longitude at that point, along with a direct link to navigate there in Google Maps. All layers are independently toggleable from the legend panel in the upper right.

In [11]:
import base64
import warnings
from pathlib import Path

import branca.colormap
import folium
import folium.raster_layers
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from folium.elements import MacroElement
from jinja2 import Template
from pynhd import WaterData
from rasterio.transform import array_bounds
from rasterio.warp import Resampling, calculate_default_transform, reproject
from shapely.geometry import LineString, box as shapely_box

## Water Features from the National Hydrography Dataset

The NHD is USGS's authoritative dataset for US surface water — streams, rivers, lakes, and ponds — at 1:24,000 scale. For GSMNP, this means the full network of named and unnamed watercourses: the Little Pigeon River, Abrams Creek, Laurel Creek, and hundreds of smaller tributaries that define the park's drainage pattern.

We fetch two layers: flowlines (linear stream networks) and water bodies (polygonal lakes and impoundments). Both come from NHD Medium Resolution via `pynhd`'s `WaterData` class — part of the same HyRiver project that provided elevation and land cover data in notebook 02.

Results are cached to `./data/nhd/` as GeoPackages so subsequent runs skip the network request. The fetch uses a bounding box slightly larger than the park to include streams that cross the park boundary.

In [2]:
boundary = gpd.read_file('./data/nps/BOUNDARY_PY.shp').to_crs('EPSG:4326')
fetch_geom = shapely_box(*boundary.total_bounds)  # bounding box for NHD query

Path('./data/nhd').mkdir(exist_ok=True)
fl_path = Path('./data/nhd/flowlines.gpkg')
wb_path = Path('./data/nhd/waterbodies.gpkg')

if fl_path.exists():
    flowlines = gpd.read_file(fl_path)
else:
    flowlines = WaterData('nhdflowline_network').bygeom(fetch_geom, geo_crs=4326)
    flowlines.to_file(fl_path, driver='GPKG')

if wb_path.exists():
    waterbodies = gpd.read_file(wb_path)
else:
    waterbodies = WaterData('nhdwaterbody').bygeom(fetch_geom, geo_crs=4326)
    waterbodies.to_file(wb_path, driver='GPKG')

# Drop first-order (ephemeral/intermittent) channels to reduce visual clutter
if 'streamorde' in flowlines.columns:
    flowlines = flowlines[flowlines['streamorde'] >= 2].copy()

flowlines   = flowlines.to_crs('EPSG:4326')
waterbodies = waterbodies.to_crs('EPSG:4326')

print(f'Flowlines:   {len(flowlines):,}')
print(f'Water bodies: {len(waterbodies):,}')

Flowlines:   1,352
Water bodies: 16


## Reprojecting Rasters for Folium

The three result rasters from notebooks 02–04 are in EPSG:5070 (Conus Albers Equal Area). Folium's `ImageOverlay` places images using EPSG:4326 (latitude/longitude) bounds, so each raster needs to be reprojected before it can sit correctly on the basemap.

After reprojection, the float values get mapped through the same colormaps used in the static figure — YlOrRd for suitability, Blues for effort, RdPu for gap score — to produce RGBA images. Pixels outside the study extent (NaN in the source raster) are set to fully transparent so the basemap shows through.

In [9]:
def raster_to_image(path, cmap_name):
    """Reproject a single-band float raster to EPSG:4326 and return a colormapped RGBA array."""
    with rasterio.open(path) as src:
        dst_crs = 'EPSG:4326'
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds
        )
        data = np.full((height, width), np.nan, dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=data,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear,
            src_nodata=src.nodata,
            dst_nodata=np.nan,
        )

    west, south, east, north = array_bounds(height, width, transform)
    bounds = [[south, west], [north, east]]

    valid = ~np.isnan(data)
    vmax  = float(np.nanmax(data))
    norm  = np.zeros_like(data)
    norm[valid] = np.clip(data[valid] / vmax, 0, 1)

    cmap = matplotlib.colormaps[cmap_name]
    rgba = (cmap(norm) * 255).astype(np.uint8)
    rgba[~valid, 3] = 0  # transparent outside study extent

    return rgba, bounds, vmax, data  # data returned for pixel-value lookup on click


suit_img,   suit_bounds,   suit_vmax,   suit_data   = raster_to_image('./data/suitability.tif', 'YlOrRd')
effort_img, effort_bounds, effort_vmax, effort_data = raster_to_image('./data/effort.tif',      'Blues')
gap_img,    gap_bounds,    gap_vmax,    gap_data    = raster_to_image('./data/gap_score.tif',   'RdPu')

print(f'Suitability  — shape {suit_img.shape[:2]}, vmax={suit_vmax:.3f}')
print(f'Effort       — shape {effort_img.shape[:2]}, vmax={effort_vmax:.3f}')
print(f'Gap score    — shape {gap_img.shape[:2]}, vmax={gap_vmax:.3f}')

Suitability  — shape (297, 511), vmax=0.401
Effort       — shape (297, 511), vmax=0.992
Gap score    — shape (297, 511), vmax=0.377


## Elevation Contour Lines

Topographic context matters for interpreting the gap score — high-gap areas at low elevation near valley floors are ecologically different from high-gap areas on exposed ridgelines. Contour lines make that relationship readable at a glance.

The elevation band is already available from the predictor stack built in notebook 02 (band 5, reprojected from USGS 3DEP). We generate contours at 200m intervals using matplotlib, then convert the resulting paths to a GeoDataFrame of `LineString` geometries. Major contours (every 500m: 500, 1000, 1500, 2000m) are drawn slightly thicker to anchor the topographic reading.

In [14]:
# Read elevation band (band 5) from the predictor stack built in notebook 02
with rasterio.open('./data/env_stack/env_predictors.tif') as src:
    dst_crs = 'EPSG:4326'
    transform, width, height = calculate_default_transform(
        src.crs, dst_crs, src.width, src.height, *src.bounds
    )
    elev_data = np.full((height, width), np.nan, dtype=np.float32)
    reproject(
        source=rasterio.band(src, 5),
        destination=elev_data,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=transform,
        dst_crs=dst_crs,
        resampling=Resampling.bilinear,
    )

west_e, south_e, east_e, north_e = array_bounds(height, width, transform)

# Build coordinate grids for matplotlib contour — row 0 is north, so y runs north → south
x = np.linspace(west_e, east_e, width)
y = np.linspace(north_e, south_e, height)
X, Y = np.meshgrid(x, y)

# 100m intervals from 200–2000m: denser spacing reads better on a terrain map
levels = list(range(200, 2100, 100))
fig, ax = plt.subplots()
cs = ax.contour(X, Y, elev_data, levels=levels)
plt.close(fig)

records = []
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for level, segs in zip(cs.levels, cs.allsegs):
        for seg in segs:
            if len(seg) >= 2:
                records.append({'geometry': LineString(seg), 'elevation': int(level)})

contours_gdf = gpd.GeoDataFrame(records, crs='EPSG:4326')
print(f'Contour lines: {len(contours_gdf):,} segments across {len(levels)} levels')

Contour lines: 933 segments across 19 levels


## Building the Map

The Folium map layers all three raster results and all five vector features (boundary, roads, trails, streams, water bodies) onto a CartoDB Positron basemap — the same tile set used in notebook 01's occurrence maps.

Each layer is wrapped in a `FeatureGroup` so it appears as a separate toggle in the layer control. Gap score is on by default; suitability and effort are off so the map opens cleanly. All vector overlays start visible.

A custom click handler adds a popup on every map click showing the latitude/longitude and a Google Maps navigation link — useful for identifying and routing to specific high-gap locations.

In [15]:
roads  = gpd.read_file('./data/nps/GRSM_ROADS.shp').to_crs('EPSG:4326')
trails = gpd.read_file('./data/nps/GRSM_TRAILS.shp').to_crs('EPSG:4326')

# ESRI World Hillshade as default — gives immediate terrain context without competing with the
# colored raster overlays. CartoDB Positron kept as a clean label-only alternative.
m = folium.Map(location=[35.61, -83.5], zoom_start=11, tiles=None)
folium.TileLayer(
    tiles='https://services.arcgisonline.com/arcgis/rest/services/Elevation/World_Hillshade/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='ESRI Hillshade',
).add_to(m)
folium.TileLayer('CartoDB positron', name='CartoDB Positron').add_to(m)

# --- Raster layers ---
# pixelated=False enables browser bilinear interpolation — eliminates the blocky square look
for name, img, bounds, show in [
    ('Gap score',           gap_img,    gap_bounds,    True),
    ('Habitat suitability', suit_img,   suit_bounds,   False),
    ('Observer effort',     effort_img, effort_bounds, False),
]:
    fg = folium.FeatureGroup(name=name, show=show)
    folium.raster_layers.ImageOverlay(
        image=img, bounds=bounds, opacity=0.65, pixelated=False
    ).add_to(fg)
    fg.add_to(m)

# Colorbar for the default-visible raster (gap score)
branca.colormap.LinearColormap(
    ['#f7f4f9', '#e7e1ef', '#d4b9da', '#c994c7', '#df65b0', '#e7298a', '#ce1256', '#91003f'],
    vmin=0,
    vmax=round(gap_vmax, 3),
    caption=f'Detection gap score  (0 – {gap_vmax:.3f})'
).add_to(m)

# --- Vector layers ---
# Color palette chosen so each layer is immediately distinguishable:
#   Roads:   burnt orange, solid — reads as infrastructure
#   Trails:  lighter orange, long dash — dashed = unpaved/foot traffic
#   Streams: blue, solid — standard water convention
#   Contours: earthy brown, tight dash — standard topo convention, distinct from both orange families
vector_layers = [
    ('GSMNP boundary',   boundary,    {'color': '#333333', 'weight': 2,   'fillOpacity': 0}),
    ('Roads',            roads,       {'color': '#d35400', 'weight': 2.5, 'fillOpacity': 0, 'opacity': 0.9}),
    ('Trails',           trails,      {'color': '#e67e22', 'weight': 1.5, 'fillOpacity': 0,
                                       'opacity': 0.85, 'dashArray': '8,6'}),
    ('Streams & rivers', flowlines,   {'color': '#2171b5', 'weight': 1.2, 'fillOpacity': 0}),
    ('Water bodies',     waterbodies, {'color': '#2171b5', 'weight': 0.5,
                                       'fillColor': '#9ecae1', 'fillOpacity': 0.5}),
]
for name, gdf, style in vector_layers:
    fg = folium.FeatureGroup(name=name, show=True)
    folium.GeoJson(gdf[['geometry']], style_function=lambda _, s=style: s).add_to(fg)
    fg.add_to(m)

# --- Elevation contours ---
# Brown + tight dash distinguishes from trails (orange long-dash) at a glance.
# 500m index contours are solid and heavier — same convention as printed topo maps.
fg_contours = folium.FeatureGroup(name='Elevation contours', show=True)
folium.GeoJson(
    contours_gdf[['geometry', 'elevation']],
    style_function=lambda f: {
        'color':     '#8B7355',
        'weight':    1.5 if f['properties']['elevation'] % 500 == 0 else 0.6,
        'opacity':   0.7 if f['properties']['elevation'] % 500 == 0 else 0.4,
        'dashArray': ''    if f['properties']['elevation'] % 500 == 0 else '3,3',
        'fillOpacity': 0,
    }
).add_to(fg_contours)
fg_contours.add_to(m)

# --- Legend ---
def swatch(color, weight=2, dash=''):
    da = f' stroke-dasharray="{dash}"' if dash else ''
    return (f'<svg width="24" height="10" style="vertical-align:middle;margin-right:4px">'
            f'<line x1="0" y1="5" x2="24" y2="5" '
            f'style="stroke:{color};stroke-width:{weight}{da}"/></svg>')

legend_html = f"""
<div style="position:fixed;bottom:40px;left:10px;background:white;border:1px solid #bbb;
            border-radius:6px;padding:10px 14px;font-size:12px;line-height:1.8;
            z-index:9999;box-shadow:2px 2px 6px rgba(0,0,0,.25);min-width:160px;">
  <div style="font-weight:bold;margin-bottom:4px;">Map layers</div>
  {swatch('#333333', 2)} GSMNP boundary<br>
  {swatch('#d35400', 3)} Roads<br>
  {swatch('#e67e22', 2, '8,6')} Trails<br>
  {swatch('#2171b5', 2)} Streams &amp; rivers<br>
  {swatch('#8B7355', 0.8, '3,3')} Contours (100 m)<br>
  {swatch('#8B7355', 2)} Contours (500 m index)<br>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# --- Click handler: raster value lookup + AllTrails link ---
CLICK_JS = """
    {% macro script(this, kwargs) %}
    (function() {
        function decodeF32(b64, h, w) {
            var bin = atob(b64), buf = new ArrayBuffer(bin.length), u8 = new Uint8Array(buf);
            for (var i = 0; i < bin.length; i++) u8[i] = bin.charCodeAt(i);
            return { data: new Float32Array(buf), h: h, w: w };
        }
        var RASTERS = {
            gap:         decodeF32("{{ this._gap_b64 }}",    {{ this._gap_h }},    {{ this._gap_w }}),
            suitability: decodeF32("{{ this._suit_b64 }}",   {{ this._suit_h }},   {{ this._suit_w }}),
            effort:      decodeF32("{{ this._effort_b64 }}", {{ this._effort_h }}, {{ this._effort_w }}),
        };
        var BOUNDS = {
            gap:         {{ this._gap_bounds_js }},
            suitability: {{ this._suit_bounds_js }},
            effort:      {{ this._effort_bounds_js }},
        };
        function lookup(key, lat, lng) {
            var r = RASTERS[key], b = BOUNDS[key];
            var row = Math.floor((b[2] - lat) / (b[2] - b[0]) * r.h);
            var col = Math.floor((lng  - b[1]) / (b[3] - b[1]) * r.w);
            if (row < 0 || row >= r.h || col < 0 || col >= r.w) return 'n/a';
            var v = r.data[row * r.w + col];
            return v < 0 ? 'n/a' : v.toFixed(3);
        }
        {{ this._parent.get_name() }}.on('click', function(e) {
            var lat = e.latlng.lat, lng = e.latlng.lng, d = 0.02;
            var atUrl = 'https://www.alltrails.com/explore?' +
                'b_br_lat=' + (lat-d).toFixed(6) + '&b_br_lng=' + (lng+d).toFixed(6) +
                '&b_tl_lat=' + (lat+d).toFixed(6) + '&b_tl_lng=' + (lng-d).toFixed(6);
            L.popup()
                .setLatLng(e.latlng)
                .setContent(
                    '<b>' + lat.toFixed(6) + ', ' + lng.toFixed(6) + '</b><br>' +
                    'Gap score: <b>'   + lookup('gap', lat, lng)         + '</b><br>' +
                    'Suitability: <b>' + lookup('suitability', lat, lng) + '</b><br>' +
                    'Effort: <b>'      + lookup('effort', lat, lng)      + '</b><br><br>' +
                    '<a href="' + atUrl + '" target="_blank" style="color:#4caf50;">' +
                    'Explore nearby trails on AllTrails &#x2197;</a>'
                )
                .openOn({{ this._parent.get_name() }});
        });
    })();
    {% endmacro %}
"""


class ClickForNavigation(MacroElement):
    def __init__(self, suit_data, suit_bounds, effort_data, effort_bounds, gap_data, gap_bounds):
        super().__init__()

        def b64(arr):
            flat = np.where(np.isnan(arr), np.float32(-1.0), arr.astype(np.float32)).flatten()
            return base64.b64encode(flat.tobytes()).decode('ascii')

        def bounds_js(b):
            return f"[{b[0][0]}, {b[0][1]}, {b[1][0]}, {b[1][1]}]"

        self._suit_b64, self._suit_h, self._suit_w       = b64(suit_data),   *suit_data.shape
        self._effort_b64, self._effort_h, self._effort_w = b64(effort_data), *effort_data.shape
        self._gap_b64,  self._gap_h,  self._gap_w        = b64(gap_data),    *gap_data.shape

        self._suit_bounds_js   = bounds_js(suit_bounds)
        self._effort_bounds_js = bounds_js(effort_bounds)
        self._gap_bounds_js    = bounds_js(gap_bounds)

        self._template = Template(CLICK_JS)


ClickForNavigation(
    suit_data, suit_bounds, effort_data, effort_bounds, gap_data, gap_bounds
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

Path('./data/figures').mkdir(exist_ok=True)
m.save('./data/figures/detection_gap_interactive.html')
print('Saved: ./data/figures/detection_gap_interactive.html')

Saved: ./data/figures/detection_gap_interactive.html


## Using the Map

The layer control in the upper right toggles any overlay on or off. Gap score is on by default — enable habitat suitability or observer effort to compare them directly. The GSMNP boundary, roads, trails, streams, and water bodies are all on by default and can be toggled individually.

Click anywhere on the map to see the exact coordinates of that point and open a Google Maps navigation link. This is most useful when you've identified a high-gap area at a specific trail intersection or ridgeline and want to route to the nearest trailhead.

The file saved to `./data/figures/detection_gap_interactive.html` is self-contained — it can be opened in any browser without a running notebook server and shared as a standalone file.

In [ ]:
m